# Feature Engineering

## 1. Objective
The objective of this notebook is to transform raw customer data into a structured, model-ready dataset. This includes encoding categorical variables, handling skewed numerical features, addressing multicollinearity, and preparing features for predictive modling. 

## 2. Load Cleaned Dataset

In [1]:
import pandas as pd

df = pd.read_csv ("../data/processed/churn_clean.csv")

df.shape, df.head()

((7043, 21),
    gender  SeniorCitizen Partner Dependents  tenure PhoneService  \
 0  Female              0     Yes         No       1           No   
 1    Male              0      No         No      34          Yes   
 2    Male              0      No         No       2          Yes   
 3    Male              0      No         No      45           No   
 4  Female              0      No         No       2          Yes   
 
       MultipleLines InternetService OnlineSecurity OnlineBackup  ...  \
 0  No phone service             DSL             No          Yes  ...   
 1                No             DSL            Yes           No  ...   
 2                No             DSL            Yes          Yes  ...   
 3  No phone service             DSL            Yes           No  ...   
 4                No     Fiber optic             No           No  ...   
 
   TechSupport StreamingTV StreamingMovies        Contract PaperlessBilling  \
 0          No          No              No  Month-to

## 3. Data Validation Checks

In [2]:
df.isna().mean().sort_values(ascending=False).head(10)

TotalCharges       0.001562
SeniorCitizen      0.000000
Partner            0.000000
Dependents         0.000000
gender             0.000000
tenure             0.000000
PhoneService       0.000000
InternetService    0.000000
MultipleLines      0.000000
OnlineBackup       0.000000
dtype: float64

The dataset contains minimal missing values, limited to the TotalCharges feature (0.16%). This missingness is likely due to customers with very short tenure who have not yet accrued billing charges. These values were imputed using the median to preserve distributional stability. 

In [3]:
df['TotalCharges'] = df["TotalCharges"].fillna(df["TotalCharges"].median())

In [4]:
df.isna().sum().sum()

np.int64(0)

In [5]:
df.dtypes

gender               object
SeniorCitizen         int64
Partner              object
Dependents           object
tenure                int64
PhoneService         object
MultipleLines        object
InternetService      object
OnlineSecurity       object
OnlineBackup         object
DeviceProtection     object
TechSupport          object
StreamingTV          object
StreamingMovies      object
Contract             object
PaperlessBilling     object
PaymentMethod        object
MonthlyCharges      float64
TotalCharges        float64
Churn                object
Churn_binary          int64
dtype: object

## 4. Separate target and Features

The target variable "Churn" is separated from the feature set to prevent data leakage and to support independent feature transformations.

In [6]:
target = "Churn_binary"
X = df.drop(columns=[target])
y = df[target].astype(int)

X.shape, y.value_counts(normalize=True)

((7043, 20),
 Churn_binary
 0    0.73463
 1    0.26537
 Name: proportion, dtype: float64)

## 5. Feature Type Identification

Features are categorized into numerical and categorical groups to apply appropriate preprocessing techniques such as scaling and encoding.

In [7]:
num_cols = X.select_dtypes(include=["number"]).columns.tolist()
cat_cols = X.select_dtypes(include=["object", "category", "bool"]).columns.tolist()

num_cols, cat_cols

(['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges'],
 ['gender',
  'Partner',
  'Dependents',
  'PhoneService',
  'MultipleLines',
  'InternetService',
  'OnlineSecurity',
  'OnlineBackup',
  'DeviceProtection',
  'TechSupport',
  'StreamingTV',
  'StreamingMovies',
  'Contract',
  'PaperlessBilling',
  'PaymentMethod',
  'Churn'])

## 6. Encode Categorical Variables

In [10]:
internet_dependent_cols = ["OnlineSecurity", "OnlineBackup","DeviceProtection","TechSupport","StreamingTV","StreamingMovies"]

for c in internet_dependent_cols:
    X[c] = X[c].replace("No internet service","No")

In [11]:
X_encoded = pd.get_dummies(X,columns=cat_cols, drop_first=True)
X_encoded.head()


,SeniorCitizen,tenure,MonthlyCharges,TotalCharges,gender_Male,Partner_Yes,Dependents_Yes,PhoneService_Yes,MultipleLines_No phone service,MultipleLines_Yes,...,TechSupport_Yes,StreamingTV_Yes,StreamingMovies_Yes,Contract_One year,Contract_Two year,PaperlessBilling_Yes,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check,Churn_Yes
0,0,1,29.85,29.85,False,True,False,False,True,False,...,False,False,False,False,False,True,False,True,False,False
1,0,34,56.95,1889.50,True,False,False,True,False,False,...,False,False,False,True,False,False,False,False,True,False
2,0,2,53.85,108.15,True,False,False,True,False,False,...,False,False,False,False,False,True,False,False,True,True
3,0,45,42.30,1840.75,True,False,False,False,True,False,...,True,False,False,True,False,False,False,False,False,False
4,0,2,70.70,151.65,False,False,False,True,False,False,...,False,False,False,False,False,True,False,True,False,True


## 7. Multicollinearity Analysis 

Multicollinearity occurs when tow or more features convey overlapping or reduntant information. This can lead to unstable coefficient estimates nad reduced interpretability in linear models such as logistic regression.

To identify potential multicollinearity, correlation analysis is performed on the feature set after encoding. Particular attention is given to numerical features that are expected to be related throgh business logic, such as tenure and billing-related variabes. 

In [12]:
import numpy as np
#Compute correlation matrix for numeric features
corr_matrix = X_encoded.corr(numeric_only=True)

# Extract high-correlation pairs (upper triangle only)
high_corr_pairs = (
    corr_matrix
        .where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
        .stack()
        .sort_values(ascending=False)
)

high_corr_pairs.head(15)

tenure           TotalCharges                   0.825464
MonthlyCharges   InternetService_Fiber optic    0.787066
                 TotalCharges                   0.650864
                 StreamingTV_Yes                0.629603
                 StreamingMovies_Yes            0.627429
tenure           Contract_Two year              0.558533
StreamingTV_Yes  StreamingMovies_Yes            0.533094
TotalCharges     DeviceProtection_Yes           0.522374
                 StreamingMovies_Yes            0.519884
                 StreamingTV_Yes                0.515279
                 OnlineBackup_Yes               0.509607
MonthlyCharges   MultipleLines_Yes              0.490434
                 DeviceProtection_Yes           0.482692
TotalCharges     MultipleLines_Yes              0.468705
Partner_Yes      Dependents_Yes                 0.452676
dtype: float64

### Interpretation of Correlation Results

After consolidating internet-dependent features, the remaining correlations reflect expected customer behavior rather than data issues.

The strongest relationship is between **tenure** and **TotalCharges**, which is expected because total charges increase as customers remain subscribed for longer periods. This indicates overlapping information between these features.

Monthly charges are moderately correlated with certain service features, such as fiber optic internet and streaming services, reflecting differences in pricing tiers and bundled offerings.

Correlations between streaming and add-on services suggest common subscription patterns, not redundancy. These features capture distinct customer choices and remain informative.

To reduce redundancy and improve stability in linear models, **TotalCharges will be excluded**, while **tenure** and **MonthlyCharges** will be retained. All features will be kept for tree-based models, which are robust to correlated inputs.


## 8. Model-Specific Feature Preparation

Different machine learning models have different requirements for input features.  
To ensure optimal performance and interpretability, two feature sets are prepared:

- A dataset for **tree-based models**, which do not require feature scaling and are robust to correlated features
- A dataset for **linear models**, which benefit from scaled numerical features and reduced multicollinearity

### Tree-Based Model Features

Tree-based models such as Decision Trees, Random Forests, and Gradient Boosting do not require feature scaling. These models can naturally handle non-linear relationships and correlated features.


In [13]:
X_tree = X_encoded.copy()
X_tree.shape


(7043, 25)

### Linear Model Features

Linear models such as Logistic Regression are sensitive to feature scale and multicollinearity. To improve model stability and interpretability, numerical features are standardized and redundant features are removed.


In [14]:
from sklearn.preprocessing import StandardScaler

X_linear = X_encoded.copy()
scaler = StandardScaler()

# Scale original numerical features
scale_cols = [c for c in ["tenure", "MonthlyCharges"] if c in X_linear.columns]
X_linear[scale_cols] = scaler.fit_transform(X_linear[scale_cols])

X_linear.shape


(7043, 25)

### Multicollinearity Reduction for Linear Models

Based on correlation analysis, `TotalCharges` contains overlapping information with tenure and monthly charges. To reduce redundancy and improve coefficient stability, this feature is removed from the linear model feature set.


In [15]:
X_linear_reduced = X_linear.copy()

if "TotalCharges" in X_linear_reduced.columns:
    X_linear_reduced = X_linear_reduced.drop(columns=["TotalCharges"])

X_linear_reduced.shape


(7043, 24)

### Save Final Feature Sets

The prepared feature sets and target variable are saved to the `data/processed` directory. This ensures reproducibility and allows the modeling phase to focus solely on training and evaluation.


In [16]:
import os
os.makedirs("../data/processed", exist_ok=True)

X_tree.to_csv("../data/processed/X_tree.csv", index=False)
X_linear_reduced.to_csv("../data/processed/X_linear.csv", index=False)
y.to_csv("../data/processed/y.csv", index=False)


### Feature Engineering Complete

At this stage, the data has been transformed into clean, model-ready feature sets.  
The next step is to train and evaluate predictive models using these features.
